In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import tqdm
from torch.utils.tensorboard import SummaryWriter

c:\Source\TEAM-PJ-DEEP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_train = pd.read_csv(('../data/train_multilabel.csv'))
df_val = pd.read_csv(('../data/test_multilabel.csv'))
df_test = pd.read_csv(('../data/valid_multilabel.csv'))


In [3]:
label_cols = df_train.columns[6:].tolist()

In [4]:
#데이터 셋 만들기(멀티라벨 용으로)

class ComplainDataset(Dataset):
    def __init__(self, df, tokenizer, label_cols, max_length=128):
        self.df = df
        self.tokenizer = tokenizer
        self.label_cols = label_cols
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx,'text']
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        label = self.df.loc[idx, self.label_cols].values.astype("float32")

        item = {
            "input_ids": encoding['input_ids'].squeeze(0),
            "attention_mask": encoding['attention_mask'].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float32) 
        }
        return item

In [5]:
# 한국어 전용 토크나이져
tokenizer = AutoTokenizer.from_pretrained("klue/roberta-base")

In [6]:
#dataset 만들기
train_dataset = ComplainDataset(df_train, tokenizer, label_cols)
val_dataset = ComplainDataset(df_val, tokenizer, label_cols)
test_dataset = ComplainDataset(df_test, tokenizer, label_cols)

In [7]:
#data_loaders 만들기
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [8]:
# 모델 생성
model = AutoModelForSequenceClassification.from_pretrained(
    "klue/roberta-base",
    num_labels=len(label_cols),
    problem_type="multi_label_classification"
)

for params in model.parameters():
    params.requires_grad = False

for param in model.roberta.encoder.layer[11].parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True
    
for name,module in model.named_parameters():
    print(name, module.requires_grad)

model

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1015.93it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Co

roberta.embeddings.word_embeddings.weight False
roberta.embeddings.token_type_embeddings.weight False
roberta.embeddings.LayerNorm.weight False
roberta.embeddings.LayerNorm.bias False
roberta.embeddings.position_embeddings.weight False
roberta.encoder.layer.0.attention.self.query.weight False
roberta.encoder.layer.0.attention.self.query.bias False
roberta.encoder.layer.0.attention.self.key.weight False
roberta.encoder.layer.0.attention.self.key.bias False
roberta.encoder.layer.0.attention.self.value.weight False
roberta.encoder.layer.0.attention.self.value.bias False
roberta.encoder.layer.0.attention.output.dense.weight False
roberta.encoder.layer.0.attention.output.dense.bias False
roberta.encoder.layer.0.attention.output.LayerNorm.weight False
roberta.encoder.layer.0.attention.output.LayerNorm.bias False
roberta.encoder.layer.0.intermediate.dense.weight False
roberta.encoder.layer.0.intermediate.dense.bias False
roberta.encoder.layer.0.output.dense.weight False
roberta.encoder.layer.

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [ ]:
writer = SummaryWriter()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.BCEWithLogitsLoss()

model.to(device)

best_val_loss = 100000000
epochs = 30
early_stop_epochs = 5
early_stop_counter = 0
count = 0

for epoch in range(epochs):
    train_tqdm = tqdm.tqdm(train_loader)
    model.train()
    train_loss_sum = 0

    for batch in train_tqdm:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss_sum += loss.item()

        writer.add_scalar("Loss/train_step", loss.item(), count)
        count += 1

        train_tqdm.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = train_loss_sum / len(train_loader)
    print("avg_train_loss",avg_train_loss)

    model.eval()    
    all_preds = []
    all_labels = []
    val_loss_sum = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            loss = criterion(logits, labels)
            val_loss_sum += loss.item()    

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

        avg_val_loss = val_loss_sum / len(val_loader)
    print( "avg_val_loss",avg_val_loss)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "multiLabel_best_model.pt")
        early_stop_counter = 0
    else:
        early_stop_counter += 1

        if early_stop_counter >= early_stop_epochs:
            print("Early stopping triggered.")
            break

    

    



100%|██████████| 505/505 [00:19<00:00, 25.43it/s, loss=0.1368]


avg_train_loss 0.2823771371994868


In [28]:
model.eval()

test_loss_sum = 0
all_preds = []
all_labels = []
test_tqdm = tqdm.tqdm(test_loader, desc="Test")
with torch.no_grad():
    for batch in test_tqdm:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        loss = criterion(logits, labels)
        test_loss_sum += loss.item()
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        test_tqdm.set_postfix(loss=f"{loss.item():.4f}")
avg_test_loss = test_loss_sum / len(test_loader)

avg_test_loss

Test: 100%|██████████| 127/127 [00:04<00:00, 29.73it/s, loss=0.0084]


0.007016862474939251

In [29]:
text = "확정일자 받고싶어"

model.eval()

encoding = tokenizer(
    text,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)
input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
pred_labels = []
for col, prob in zip(label_cols, probs):
    if prob >= 0.3:
        pred_labels.append((col, float(prob)))
pred_labels = sorted(pred_labels, key=lambda x: x[1], reverse=True)

print(pred_labels)
print(probs)

[('기관:읍·면·동 행정복지센터', 0.9599643349647522), ('문서:신분증', 0.888337254524231), ('부서:민원행정팀', 0.842357873916626), ('민원명:확정일자 신청', 0.6383340358734131), ('문서:확정일자 신청서', 0.6376564502716064), ('문서:임대차계약서 원본', 0.5909386277198792)]
[4.2654807e-04 2.7386865e-03 8.2693445e-03 7.1817632e-03 4.2322703e-04
 6.9621234e-04 1.3953535e-03 8.9131147e-03 9.0407841e-03 3.0140439e-03
 5.5906987e-03 9.5996433e-01 3.9337627e-03 8.0574509e-03 3.7337313e-03
 4.7452608e-03 1.1302728e-02 6.4891702e-03 1.1079236e-03 2.6974641e-03
 1.8264533e-03 8.0215588e-04 2.1135854e-02 1.1553493e-03 2.2404245e-03
 9.8211057e-03 1.0428329e-03 7.5767556e-04 2.8040600e-03 4.7208490e-03
 6.4698099e-03 3.3861428e-04 3.4880254e-03 1.1872625e-03 3.6954714e-03
 3.0752558e-03 8.0243647e-03 2.2528186e-02 3.7791368e-03 6.9855368e-03
 1.5863321e-03 3.8437180e-03 5.2627537e-04 9.0121658e-04 9.8400330e-04
 1.0018594e-03 4.6755243e-04 3.4808056e-04 3.4272452e-03 4.4012601e-03
 8.9703433e-02 2.5819477e-03 2.2817468e-03 6.6567695e-04 9.1689394e-04
 